# Rhyme manifold inspection

After running `06_communities_and_manifolds.py`, use this notebook to inspect the rhyme manifold:
- Which community contains Goodfire's 23 features?
- What are the autointerp labels?
- What does the principal curve look like in PCA-3D space?
- What's the NMI vs. Goodfire's anchor list?

In [ ]:
import sys; sys.path.insert(0, '../src')
from neograph.cypher import NeographClient
from neograph.evals import nmi_vs_goodfire, rhyme_community_summary, GOODFIRE_RHYME_FEATURES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

c = NeographClient()

In [ ]:
# 1. Where did Goodfire's 23 features end up?
rhyme_community_summary(c)

In [ ]:
# 2. Inspect the top community in detail
summary = rhyme_community_summary(c)
top_cid = summary.get('top_community_id')
if top_cid is not None:
    df = c.run_df('''
        MATCH (f:SAEFeature {communityId: $cid})-[:LIES_ON]->(m:Manifold)
        OPTIONAL MATCH (f)-[:LABELED_AS]->(a:AutoInterpLabel)
        RETURN f.index AS index, f.activation_density AS density, a.text AS label, m.id AS manifold
        ORDER BY f.activation_density DESC
    ''', cid=top_cid)
    display(df.head(30))

In [ ]:
# 3. Plot waypoints in PCA-3D (manifold's own PCA, projected to 3D for display)
wps = c.run_df('''
    MATCH (m:Manifold)-[:HAS_WAYPOINT]->(w:Waypoint)
    WHERE m.id CONTAINS 'rhyme' OR m.id CONTAINS toString($cid)
    RETURN m.id AS mid, w.index AS wi, w.arc_position AS arc, w.centroid AS centroid
    ORDER BY m.id, w.index
''', cid=str(top_cid) if top_cid is not None else '0')
wps.head()

In [ ]:
from sklearn.decomposition import PCA
if len(wps):
    centroids = np.stack(wps['centroid'].values)
    pca = PCA(n_components=3).fit(centroids)
    proj = pca.transform(centroids)
    fig = plt.figure(figsize=(7,7))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(proj[:,0], proj[:,1], proj[:,2], '-o', c='r')
    for i, p in enumerate(proj):
        ax.text(p[0], p[1], p[2], str(int(wps['wi'].iloc[i])))
    ax.set_title(wps['mid'].iloc[0])
    plt.show()

In [ ]:
# 4. NMI raw
nmi_vs_goodfire(c)